In [ ]:
## find cases that are hits according to Turbo eval but non-inferables according to 4.1

In [7]:
import pandas as pd
import re

In [2]:
eval41 = pd.read_csv('41eval-GPT41Prod.csv', index_col = 0)

In [3]:
evalturbo = pd.read_csv('GPT41ProdDiagandEval.csv', index_col = 0)

In [4]:
def analyze_results(text, index):
    mistakes = []
    hits = []
    excluded = [] #not a medical diagnosis
    noninferables = []
    current = 1
    total_adjust = 0
    #for conditions that GPT-4 grouped together - still doesn't capture issue with hadmid 23707730
    from_nums = []
    to_nums = []
    grouped = re.findall(r'\n\d+-\d+:', text)
    if len(grouped) > 0:
        #print("Grouping found!")
        #print("At index: ")
        #print(index)
        for elem in grouped:
            from_nums.append(str(int(elem.split('-')[0]))) #str(int()) for safety
            to_nums.append(str(int(elem.split('-')[1].strip(':'))))
    while 1:
        try:
            if current == 1:
                number = str(current)
                #sometimes GPT-4 adds words like Diagnosis or Actual diagnosis, and we want to capture that
                pre_word = text.split(number, 1)[0]
            #for conditions that GPT-4 grouped together
            elif str(current) in from_nums:
                idx = from_nums.index(str(current))
                number = grouped[idx]
                total_adjust += int(to_nums[idx]) - current
                current = int(to_nums[idx])
            else:
                number = '\n' + pre_word + str(current)
            nextOne = '\n' + pre_word + str(current+1)
            if text.split(number, 1)[1].split('Question 1: ', 1)[1][:2] == 'No':
                if 'not a medical diagnosis' in text.split(number, 1)[1].split('Question 1: ', 1)[1].split('Question 2: ', 1)[0].split(nextOne, 1)[0]:
                    print(index)
                    print(text.split(number, 1)[1].split('Question 1: ', 1)[0])
                    excluded.append(str(current))
                else:
                    hits.append(str(current))
            elif text.split(number, 1)[1].split('Question 2: ', 1)[1][:3] == 'Yes':
                mistakes.append(str(current))
            elif text.split(number, 1)[1].split('Question 2: ', 1)[1][:2] == 'No':
                noninferables.append(str(current))
            else:
                print("Unable to parse text when looking at diagnosis number: ")
                print(current)
                print("At index: ")
                print(index)
        except:
            #print('Diagnosis number not found in text: ')
            #print(current)
            total = current - 1 - total_adjust
            break
        current += 1
    return pd.Series([len(hits), len(noninferables), len(mistakes), len(excluded), '; '.join(hits), '; '.join(noninferables), '; '.join(mistakes), '; '.join(excluded), total])

In [5]:
def analysis(result_df):
  analyzed_df = result_df.apply(lambda row: analyze_results(row['GPT-Eval'], row.name),1)
  analyzed_df.columns = ['no_hits', 'no_noninferables', 'no_mistakes', 'no_excluded', 'hits', 'noninferables', 'mistakes', 'excluded', 'total_ICD_diagnoses']
  analyzed_df['error'] = analyzed_df['no_mistakes'] / (analyzed_df['no_hits'] + analyzed_df['no_mistakes'])
  analyzed_df['sensitivity'] = 1-analyzed_df['error']
  print(analyzed_df['sensitivity'].mean())
  print(1-(analyzed_df['no_mistakes'].sum() / (analyzed_df['no_hits'].sum() + analyzed_df['no_mistakes'].sum())))

  results = pd.concat([result_df, analyzed_df], axis=1)

  return results

In [8]:
eval41 = analysis(eval41)

6
: Encounter for examination for normal comparison and control in clinical research program  

8
: Physical restraint status  

14
: Unspecified place or not applicable  

16
: Other place in single-family (private) house as the place of occurrence of the external cause  

16
: Physical restraint status  

16
: Family history of alcohol abuse and dependence  

16
: Family history of other mental and behavioral disorders  

16
: Exposure to other specified factors, initial encounter  

16
: Unspecified place or not applicable  

19
: Examination of participant in clinical trial  

19
: Accidents occurring in residential institution  

22
: Encounter for examination for normal comparison and control in clinical research program  

24
: Other specified places as the place of occurrence of the external cause  

28
: Encounter for immunization  

34
: Encounter for palliative care  

37
: Unspecified place in unspecified non-institutional (private) residence as the place of occurrence of t

In [9]:
evalturbo = analysis(evalturbo)

0
: Long-term (current) use of aspirin

14
: Unspecified place or not applicable

37
: Unspecified place in unspecified non-institutional (private) residence as the place of occurrence of the external cause

38
: Patient room in hospital as the place of occurrence of the external cause

47
: Other place in hospital as the place of occurrence of the external cause

72
: Unspecified place or not applicable

88
: Examination of participant in clinical trial

102
: Examination of participant in clinical trial

105
: Personal history of antineoplastic chemotherapy

105
: Personal history of tobacco use

105
: Do not resuscitate status

105
: Encounter for palliative care

108
: Osteoarthrosis, localized, not specified whether primary or secondary, site unspecified

108
: Do not resuscitate status

108
: Long-term (current) use of steroids

108
: Long-term (current) use of aspirin

108
: Physical restraints status

137
: Do not resuscitate

182
: Personal history of tobacco use

185
: Do not

In [11]:
#merge and find the patients with differences in no_noninferables, order by highest number of diffs to lowest

In [24]:
evalturbo.columns

Index(['hadm_id', 'diagnoses', 'GPT_input', 'GPT-Diagnoses', 'GPT-Eval',
       'no_hits', 'no_noninferables', 'no_mistakes', 'no_excluded', 'hits',
       'noninferables', 'mistakes', 'excluded', 'total_ICD_diagnoses', 'error',
       'sensitivity'],
      dtype='object')

In [30]:
evalturbo['diagnoses']

,diagnoses
0,1:Coronary atherosclerosis of native coronary ...
1,"1:Sepsis, unspecified organism\r\n2:Other panc..."
2,1:Mitral valve disorders\r\n2:Rupture of chord...
3,1:Benign neoplasm of ovary\r\n2:Pulmonary coll...
4,"1:Atrioventricular block, complete\r\n2:Chroni..."
...,...
995,1:Other chest pain\r\n2:Atherosclerotic heart ...
996,"1:Pneumonia, organism unspecified\r\n2:Acute p..."
997,1:Coronary atherosclerosis of native coronary ...
998,"1:Pneumonia, organism unspecified\r\n2:Liver r..."


In [46]:
evalturbo['diagnoses'] = evalturbo['diagnoses'].str.replace('\r', '', regex=False)
eval41['diagnoses'] = eval41['diagnoses'].str.replace('\r', '', regex=False)

evalturbo['GPT_input'] = evalturbo['GPT_input'].str.replace('\r', '', regex=False)
eval41['GPT_input'] = eval41['GPT_input'].str.replace('\r', '', regex=False)

In [47]:
merged = evalturbo.merge(eval41, how='outer', on=['hadm_id', 'diagnoses', 'GPT_input'], suffixes=('_turbo', '_41'))

In [49]:
merged['no_noninferables_diff'] =  merged['no_noninferables_41'] - merged['no_noninferables_turbo']

In [50]:
merged = merged.sort_values(by='no_noninferables_diff', ascending=False)

In [51]:
merged[['noninferables_turbo', 'noninferables_41']]

,noninferables_turbo,noninferables_41
352,,2; 6; 7; 8; 9; 10; 11; 14; 16; 19; 22; 23; 24;...
819,,4; 8; 11; 13; 14; 15; 16; 17; 18; 20; 21; 22; ...
558,12; 15; 17; 18; 21,1; 3; 4; 5; 6; 7; 8; 9; 10; 12; 14; 15; 17; 18...
60,,1; 7; 9; 10; 12; 13; 14; 15; 16; 18; 19
189,9; 10; 15,2; 4; 5; 6; 7; 8; 9; 10; 11; 12; 13; 14; 15; 16
...,...,...
687,5; 8; 9; 10; 16; 17; 18; 19; 20; 21,5; 8; 16; 17
275,1; 2; 3; 4; 5; 6; 7; 8; 9; 10; 11; 12; 13; 14;...,1; 2; 3; 4; 5; 6; 7; 9; 13; 14; 15; 16; 17; 18...
843,1; 3; 5; 7; 8; 9; 11; 12; 13; 14; 16; 17; 18; ...,1; 3; 5; 7; 11; 15; 16; 19; 20
575,2; 6; 10; 11; 12; 14; 15; 16,2


In [52]:
merged.columns

Index(['hadm_id', 'diagnoses', 'GPT_input', 'GPT-Diagnoses_turbo',
       'GPT-Eval_turbo', 'no_hits_turbo', 'no_noninferables_turbo',
       'no_mistakes_turbo', 'no_excluded_turbo', 'hits_turbo',
       'noninferables_turbo', 'mistakes_turbo', 'excluded_turbo',
       'total_ICD_diagnoses_turbo', 'error_turbo', 'sensitivity_turbo',
       'GPT-Diagnoses_41', 'GPT-Eval_41', 'no_hits_41', 'no_noninferables_41',
       'no_mistakes_41', 'no_excluded_41', 'hits_41', 'noninferables_41',
       'mistakes_41', 'excluded_41', 'total_ICD_diagnoses_41', 'error_41',
       'sensitivity_41', 'no_noninferables_diff'],
      dtype='object')

In [54]:
# prompt: in merged you have two columns that only contain numbers separated by ;
# 1. noninferables_turbo
# 2. noninferables_41
# Make them into sets of integers, remove the intersection from noninferables_41 to make it a column called noninferables_41_only and use it to index the column called diagnoses -> the diagnoses that correspond to those indices should be printed in column noninferable_diagnoses_41_only

merged['noninferables_turbo_set'] = merged['noninferables_turbo'].apply(lambda x: set(map(int, x.split('; '))) if isinstance(x, str) and x != '' else set())
merged['noninferables_41_set'] = merged['noninferables_41'].apply(lambda x: set(map(int, x.split('; '))) if isinstance(x, str) and x != '' else set())

merged['noninferables_41_only'] = merged.apply(lambda row: list(row['noninferables_41_set'] - row['noninferables_turbo_set']), axis=1)

def get_diagnoses_from_indices(diagnoses_str, indices_list):
    if not isinstance(diagnoses_str, str):
        return ""
    diagnoses_list = diagnoses_str.split('\n')
    selected_diagnoses = []
    for index in indices_list:
        # Adjust index for 0-based list if needed, assuming diagnosis numbers are 1-based
        list_index = index - 1
        if 0 <= list_index < len(diagnoses_list):
            selected_diagnoses.append(diagnoses_list[list_index].strip())
    return '; '.join(selected_diagnoses)

merged['noninferable_diagnoses_41_only'] = merged.apply(lambda row: get_diagnoses_from_indices(row['diagnoses'], row['noninferables_41_only']), axis=1)

In [58]:
merged[['GPT_input', 'noninferables_41_only', 'noninferable_diagnoses_41_only']]

,GPT_input,noninferables_41_only,noninferable_diagnoses_41_only
352,Blood report: \nThe patient stayed in the hosp...,"[2, 6, 7, 8, 9, 10, 11, 14, 16, 19, 22, 23, 24...",2:Other encephalopathy; 6:Stricture and stenos...
819,Blood report: \nThe patient stayed in the hosp...,"[4, 8, 11, 13, 14, 15, 16, 17, 18, 20, 21, 22,...",4:Hyperosmolality and/or hypernatremia; 8:Deli...
558,Blood report: \nThe patient stayed in the hosp...,"[1, 3, 4, 5, 6, 7, 8, 9, 10, 14, 20, 22]",1:Acute and subacute hepatic failure without c...
60,Blood report: \nThe patient stayed in the hosp...,"[1, 7, 9, 10, 12, 13, 14, 15, 16, 18, 19]","1:Fever, unspecified; 7:Displaced bicondylar f..."
189,Blood report: \nThe patient stayed in the hosp...,"[2, 4, 5, 6, 7, 8, 11, 12, 13, 14, 16]",2:Chronic systolic (congestive) heart failure;...
...,...,...,...
687,Blood report: \nThe patient stayed in the hosp...,[],
275,Blood report: \nThe patient stayed in the hosp...,[],
843,Blood report: \nThe patient stayed in the hosp...,[15],15:Unspecified viral hepatitis C without hepat...
575,Blood report: \nThe patient stayed in the hosp...,[],


In [124]:
# prompt: # prompt: in merged you have two columns that only contain numbers separated by ;
# # 1. hits_turbo
# # 2. noninferables_41
# Make them into sets of integers, find the intersection (put it in a column called hits_turbo_noninferables_41_intersect and use this to index the column called diagnoses -> the diagnoses that correspond to those indices should be printed in column intersect_diags

merged['hits_turbo_set'] = merged['hits_turbo'].apply(lambda x: set(map(int, x.split('; '))) if isinstance(x, str) and x != '' else set())

merged['hits_turbo_noninferables_41_intersect'] = merged.apply(lambda row: list(row['hits_turbo_set'].intersection(row['noninferables_41_set'])), axis=1)

merged['intersect_diags'] = merged.apply(lambda row: get_diagnoses_from_indices(row['diagnoses'], row['hits_turbo_noninferables_41_intersect']), axis=1)

# Display the relevant columns
merged[['hadm_id', 'hits_turbo_noninferables_41_intersect', 'intersect_diags']].head(20)

,hadm_id,hits_turbo_noninferables_41_intersect,intersect_diags
352,23685637,"[2, 6, 7, 8, 9, 10, 11, 14, 16, 19, 22, 23, 24, 25, 26, 27, 28, 29]","2:Other encephalopathy; 6:Stricture and stenosis of esophagus; 7:Chronic systolic heart failure; 8:Acidosis; 9:Unspecified protein-calorie malnutrition; 10:Stridor; 11:Dysphagia, unspecified; 14:Malignant neoplasm of bladder, part unspecified; 16:Malignant neoplasm of prostate; 19:Congestive heart failure, unspecified; 22:Hypertensive chronic kidney disease, unspecified, with chronic kidney disease stage I through stage IV, or unspecified; 23:Chronic kidney disease, unspecified; 24:Stricture or kinking of ureter; 25:Hematuria, unspecified; 26:Iron deficiency anemia secondary to blood loss (chronic); 27:Altered mental status; 28:Diarrhea; 29:Other drugs and medicinal substances causing adverse effects in therapeutic use"
819,28339862,"[4, 8, 11, 13, 15, 16, 17, 18, 20, 21, 22, 26, 27]","4:Hyperosmolality and/or hypernatremia; 8:Delirium due to conditions classified elsewhere; 11:Chronic airway obstruction, not elsewhere classified; 13:Personal history of transient ischemic attack (TIA), and cerebral infarction without residual deficits; 15:Personal history of malignant neoplasm of bronchus and lung; 16:Acquired absence of organ, lung; 17:Dementia, unspecified, without behavioral disturbance; 18:Coronary atherosclerosis of native coronary artery; 20:Pure hypercholesterolemia; 21:Unspecified essential hypertension; 22:Personal history of tobacco use; 26:Peripheral vascular disease, unspecified; 27:Esophageal reflux"
558,25805325,"[1, 3, 4, 5, 6, 7, 8, 9, 10, 14, 20, 22]","1:Acute and subacute hepatic failure without coma; 3:Toxic encephalopathy; 4:Liver transplant failure; 5:Nephrotic syndrome with focal and segmental glomerular lesions; 6:Malignant neoplasm associated with transplanted organ; 7:Post-transplant lymphoproliferative disorder (PTLD); 8:Diffuse large B-cell lymphoma, lymph nodes of multiple sites; 9:Other complications of liver transplant; 10:Hepatic failure, unspecified without coma; 14:Essential (primary) hypertension; 20:Other cirrhosis of liver; 22:Sequelae of viral hepatitis"
60,20760364,"[1, 7, 9, 10, 12, 13, 14, 15, 16, 18, 19]","1:Fever, unspecified; 7:Displaced bicondylar fracture of right tibia, initial encounter for closed fracture; 9:Chronic systolic (congestive) heart failure; 10:Fall from chair, initial encounter; 12:Dependence on renal dialysis; 13:Long term (current) use of insulin; 14:Ischemic cardiomyopathy; 15:Atherosclerotic heart disease of native coronary artery without angina pectoris; 16:Presence of aortocoronary bypass graft; 18:Hyperlipidemia, unspecified; 19:Obstructive sleep apnea (adult) (pediatric)"
189,22068465,"[2, 4, 5, 6, 7, 8, 11, 12, 13, 14, 16]","2:Chronic systolic (congestive) heart failure; 4:Personal history of nicotine dependence; 5:Atherosclerotic heart disease of native coronary artery without angina pectoris; 6:Presence of aortocoronary bypass graft; 7:Type 2 diabetes mellitus with diabetic peripheral angiopathy without gangrene; 8:Hyperlipidemia, unspecified; 11:Other long term (current) drug therapy; 12:Long term (current) use of oral hypoglycemic drugs; 13:Long term (current) use of aspirin; 14:Hypertensive heart disease with heart failure; 16:Old myocardial infarction"
683,27092887,"[1, 2, 3, 7, 8, 10, 11, 12, 13, 14, 17]","1:Persistent atrial fibrillation; 2:Cardiac tamponade; 3:Acute pericarditis, unspecified; 7:Other sequelae of cerebral infarction; 8:Heteronymous bilateral field defects; 10:Hyperlipidemia, unspecified; 11:Personal history of nicotine dependence; 12:Family history of ischemic heart disease and other diseases of the circulatory system; 13:Long term (current) use of anticoagulants; 14:Unspecified atrial flutter; 17:Ataxia following cerebral infarction"
587,26038786,"[1, 3, 5, 7, 10, 13, 14, 15, 16, 17, 19]","1:Necrosis of amputation stump, left lower extremity; 3:Non-pressure chronic ulcer of right heel and 

In [135]:
chosen_index = 262

In [136]:
merged.loc[chosen_index]['intersect_diags']

'3:Unspecified asthma with (acute) exacerbation; 4:Chronic diastolic (congestive) heart failure; 5:Body mass index (BMI) 40.0-44.9, adult; 7:Respiratory conditions due to unspecified external agent; 8:Dependence on supplemental oxygen; 9:Personal history of pulmonary embolism; 10:Long term (current) use of anticoagulants; 11:Personal history of other malignant neoplasm of large intestine; 12:Acute upper respiratory infection, unspecified; 13:Long term (current) use of insulin; 15:Morbid (severe) obesity due to excess calories'

In [137]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
print(merged.loc[chosen_index]['GPT-Eval_turbo'])

D1: Acute and chronic respiratory failure with hypercapnia
Question 1: No, this is similar to diagnosis *Chronic Type 2 Respiratory Failure (Hypercapnic Respiratory Failure)*
Question 2: N/A

D2: Interstitial pulmonary disease, unspecified
Question 1: No, this falls under the broader category of *Chronic Pulmonary Disease*
Question 2: N/A

D3: Unspecified asthma with (acute) exacerbation
Question 1: No, this could be considered as part of the differential diagnosis for *Chronic Pulmonary Disease*
Question 2: N/A

D4: Chronic diastolic (congestive) heart failure
Question 1: No, this could be related to *Chronic Pulmonary Disease* and *Metabolic Alkalosis (Compensated with Chronic Hypercapnia)* due to chronic congestion
Question 2: N/A

D5: Body mass index (BMI) 40.0-44.9, adult
Question 1: No, this is related to *Chronic Type 2 Respiratory Failure (Hypercapnic Respiratory Failure)* as severe obesity hypoventilation was considered
Question 2: N/A

D6: Acute and chronic respiratory failur

In [138]:
print(merged.loc[chosen_index]['GPT-Eval_41'])

D1: Acute and chronic respiratory failure with hypercapnia  
Question 1: No, this is the same as *Type 2 (hypercapnic) respiratory failure* in my initial list.  
Question 2: N/A

D2: Interstitial pulmonary disease, unspecified  
Question 1: No, this is covered under *Chronic or resolved lung disease/interstitial lung process* in my initial list.  
Question 2: N/A

D3: Unspecified asthma with (acute) exacerbation  
Question 1: Yes  
Question 2: No, there is no mention of a history of asthma, wheeze, reversible airway obstruction, or bronchodilator response in the data provided. Additional data such as a history of asthma, clinical wheeze, or spirometry would have been helpful.

D4: Chronic diastolic (congestive) heart failure  
Question 1: Yes  
Question 2: No, there is no direct evidence of heart failure in the data provided. The chest x-ray mentions "accentuation of the heart size" and "perihilar and bibasilar opacities," which could be suggestive, but there is no mention of clinical 

In [139]:
print(merged.loc[chosen_index]['GPT-Diagnoses_turbo'])

*Chronic Type 2 Respiratory Failure (Hypercapnic Respiratory Failure)*
- Venous blood gas: pH 7.36 (high-normal), pCO2 73 mmHg (markedly elevated), Total CO2 43 (elevated), base excess 11 (elevated)
- pO2 55 (hypoxaemia)
- Chronic compensation (normal-ish pH with elevated pCO2/base excess, consistent with chronicity)
- History of baseline dyspnoea, now acutely worsened
- Radiology: evidence of chronic cardiopulmonary changes (perihilar and basilar opacities, bronchovascular crowding, slightly reduced lung volumes)
- Clinical suspicion for underlying chronic lung disease or neuromuscular-structural disorder is high due to these features
- Alternative differentials to consider: severe obesity hypoventilation, neuromuscular disease, chest wall deformities

*Anaemia (Normocytic, Mild)*
- Minimum haemoglobin: 11 g/dL, maximum: 11.6 g/dL (below normal reference range)
- Haematocrit: minimum 35.6, maximum 38 (mildly reduced)
- MCV: normal range (89–90)
- RDW: 15.6–15.8 (mildly elevated), poss